# Setup

In [2]:
import csv
from lets_plot import *
import pandas as pd
import requests as rq
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

LetsPlot.setup_html(isolated_frame=True)

In [3]:
URL = "https://github.com/fivethirtyeight/data/raw/master/star-wars-survey/StarWars.csv"
ARTICLE = "https://fivethirtyeight.com/features/americas-favorite-star-wars-movies-and-least-favorite-characters/"

response = rq.get(URL)
content_iso = response.content.decode("ISO-8859-1", errors="ignore")
content_utf8 = content_iso.encode("utf-8", errors="ignore").decode("utf-8", errors="ignore")

CSV = "StarWars_utf8.csv"
with open(CSV, "w", encoding="utf-8") as f:
    f.write(content_utf8)

del response, content_iso, content_utf8


df = pd.read_csv(CSV, encoding="utf-8").drop(index=0).reset_index(drop=True)

In [ ]:
# df[["Have you seen any of the 6 films in the Star Wars franchise?"]].value_counts()

Have you seen any of the 6 films in the Star Wars franchise?
Yes                                                             936
No                                                              250
Name: count, dtype: int64

# Q1

In [4]:
NUM_MOVIES = 6
NUM_CHARACTERS = 14
EPISODES = {
    "Star Wars: Episode I  The Phantom Menace"     : 1,
    "Star Wars: Episode II  Attack of the Clones"  : 2,
    "Star Wars: Episode III  Revenge of the Sith"  : 3,
    "Star Wars: Episode IV  A New Hope"            : 4,
    "Star Wars: Episode V The Empire Strikes Back" : 5,
    "Star Wars: Episode VI Return of the Jedi"     : 6
}
"""
gross_col_names = [
    "RespondentID",
    "Have you seen any of the 6 films in the Star Wars franchise?",
    "Do you consider yourself to be a fan of the Star Wars film franchise?",
    "Which of the following Star Wars films have you seen? Please select all that apply.",
    "",
    "",
    "",
    "",
    "",
    "Please rank the Star Wars films in order of preference with 1 being your favorite film in the franchise and 6 being your least favorite film.",
    "",
    "",
    "",
    "",
    "",
    "\"Please state whether you view the following characters favorably, unfavorably, or are unfamiliar with him/her.\"",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "Which character shot first?",
    "Are you familiar with the Expanded Universe?",
    "Do you consider yourself to be a fan of the Expanded Universe?",
    "Do you consider yourself to be a fan of the Star Trek franchise?",
    "Gender",
    "Age",
    "Household Income",
    "Education",
    "Location (Census Region)"
]
"""
CLEAN_COL_NAMES = [
    'id',
    'seen_any',
    'is_fan',
    'seen_ep1',
    'seen_ep2',
    'seen_ep3',
    'seen_ep4',
    'seen_ep5',
    'seen_ep6',
    'ep1_rank',
    'ep2_rank',
    'ep3_rank',
    'ep4_rank',
    'ep5_rank',
    'ep6_rank',
    'han_solo_rank',
    'luke_skywalker_rank',
    'princess_leia_organa_rank',
    'anakin_skywalker_rank',
    'obi_wan_kenobi_rank',
    'emperor_palpatine_rank',
    'darth_vader_rank',
    'lando_calrissian_rank',
    'boba_fett_rank',
    'c-3p0_rank',
    'r2_d2_rank',
    'jar_jar_binks_rank',
    'padme_amidala_rank',
    'yoda_rank',
    'who_shot_first',
    'is_familiar_eu',
    'is_fan_eu',
    'is_fan_star_trek',
    'gender',
    'age',
    'income_hh',
    'edu',
    'location'
]

#### This was pointless... more work than just manually renaming columns :P

In [ ]:
with open(CSV, "r", encoding="ISO-8859-1") as f:
    reader = csv.reader(f)
    cols = next(reader)
    cols_info = next(reader)

cols_with_info = list(zip(cols, cols_info))
cols_with_info

In [ ]:
filled_cols = []
last_real_i = 0

for i, col in enumerate(cols):
    if col:
        filled_cols.append(col)
        last_real_i = i
    else:
        filled_cols.append(cols[last_real_i])

repeat_start_indices = []
repeats = set()
last_seen = filled_cols[i]
for i in range(1, len(filled_cols)):
    curr = filled_cols[i]
    if curr == last_seen and curr not in repeats:
        repeats.update([curr])
        repeat_start_indices.append(i-1)
    last_seen = curr

In [ ]:
filled_cols
# repeat_start_indices

In [ ]:
renamed = []
for i, col in enumerate(filled_cols):
    if i < repeat_start_indices[0]:
        renamed.append(col)
    elif repeat_start_indices[0] <= i < repeat_start_indices[1]:
        renamed.append(f"seen_ep{EPISODES[cols_with_info[i][1]]}")
        # renamed.append(f"seen_{EPISODES[cols_with_info[i - repeat_start_indices[0]][1]]}")
    elif repeat_start_indices[1] <= i < repeat_start_indices[2]:
        renamed.append(f"ep{EPISODES[cols_with_info[i][1]]}_rank")
        # renamed.append(f"{EPISODES[cols_with_info[i - repeat_start_indices[1]][1]]}_rank")
    elif repeat_start_indices[2] <= i < repeat_start_indices[2] + NUM_CHARACTERS:
        renamed.append(f"{cols_with_info[i][1].lower().replace(' ', '_')}_rank")
    else:
        renamed.append(col)

renamed

# Q2

#### This was actually worth something

In [5]:
BINARY = {
    "Yes"    : True,
    "No"     : False,
    "Female" : True,
    "Male"   : False
}
FAVOR = {
    "Unfamiliar (N/A)"        : 0,
    "Very unfavorably"        : 1,
    "Somewhat unfavorably"    : 2,
    "Neither favorably nor "
    + "unfavorably (neutral)" : 3,
    "Somewhat favorably"      : 4,
    "Very favorably"          : 5
}
CHARACTERS = {
    "Han Solo"             :  0,
    "Luke Skywalker"       :  1,
    "Princess Leia Organa" :  2,
    "Anakin Skywalker"     :  3,
    "Obi Wan Kenobi"       :  4,
    "Emperor Palpatine"    :  5,
    "Darth Vader"          :  6,
    "Lando Calrissian"     :  7,
    "Boba Fett"            :  8,
    "C-3P0"                :  9,
    "R2 D2"                : 10,
    "Jar Jar Binks"        : 11,
    "Padme Amidala"        : 12,
    "Yoda"                 : 13
}
SHOT_FIRST = {
    "Han"             : 0,
    "I don't understand "
    + "this question" : 1,
    "Greedo"          : 2
}
AGES = {
    "18-29" : 0,
    "30-44" : 1,
    "45-60" : 2,
    "> 60"  : 3
}
INCOME = {
    "$0 - $24,999"        : 0,
    "$25,000 - $49,999"   : 1,
    "$50,000 - $99,999"   : 2,
    "$100,000 - $149,999" : 3,
    "$150,000+"           : 4,
}
EDUCATION = {
    "Less than high school degree"     : 0,
    "High school degree"               : 1,
    "Some college or Associate degree" : 2,
    "Bachelor degree"                  : 3,
    "Graduate degree"                  : 4
}
LOCATION = {
    "East North Central" : "enc",
    "East South Central" : "esc",
    "Middle Atlantic"    : "mat",
    "Mountain"           : "mtn",
    "New England"        : "neg",
    "Pacific"            : "pcf",
    "South Atlantic"     : "sat",
    "West North Central" : "wnc",
    "West South Central" : "wsc"
}

In [6]:
binary_cols = [
    'seen_any',
    'is_fan',
    'is_familiar_eu',
    'is_fan_eu',
    'is_fan_star_trek',
    'gender'
]

In [7]:
df_mod = df.copy()
df_mod.columns = CLEAN_COL_NAMES
for col in df_mod.columns:
    if col in binary_cols:
        df_mod[col] = df_mod[col].map(BINARY)
    elif col.startswith("seen_ep"):
        df_mod[col] = df_mod[col].notna()
    elif col.startswith("ep") and col.endswith("_rank"):
        df_mod[col] = df_mod[col].astype(int, errors='ignore')
    elif col.endswith("_rank") and not col.startswith("ep"):
        df_mod[col] = df_mod[col].map(FAVOR)
    elif col == "who_shot_first":
        df_mod[col] = df_mod[col].map(SHOT_FIRST)
    elif col == "age":
        df_mod[col] = df_mod[col].map(AGES)
    elif col == "income_hh":
        df_mod[col] = df_mod[col].map(INCOME)
    elif col == "edu":
        df_mod[col] = df_mod[col].map(EDUCATION)
    elif col == "location":
        df_mod[col] = df_mod[col].map(LOCATION)

df_mod = pd.get_dummies(df_mod, columns=['location'], prefix='loc')

In [25]:
df_mod

,id,seen_any,is_fan,seen_ep1,seen_ep2,seen_ep3,seen_ep4,seen_ep5,seen_ep6,ep1_rank,...,edu,loc_enc,loc_esc,loc_mat,loc_mtn,loc_neg,loc_pcf,loc_sat,loc_wnc,loc_wsc
0,3.292880e+09,True,True,True,True,True,True,True,True,3,...,1.0,False,False,False,False,False,False,True,False,False
1,3.292880e+09,False,NaN,False,False,False,False,False,False,NaN,...,3.0,False,False,False,False,False,False,False,False,True
2,3.292765e+09,True,False,True,True,True,False,False,False,1,...,1.0,False,False,False,False,False,False,False,True,False
3,3.292763e+09,True,True,True,True,True,True,True,True,5,...,2.0,False,False,False,False,False,False,False,True,False
4,3.292731e+09,True,True,True,True,True,True,True,True,5,...,2.0,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1181,3.288389e+09,True,True,True,True,True,True,True,True,5,...,2.0,True,False,False,False,False,False,False,False,False
1182,3.288379e+09,True,True,True,True,True,True,True,True,4,...,3.0,False,False,False,True,False,False,False,False,False
1183,3.288375e+09,False,NaN,False,False,False,False,False,False,NaN,...,3.0,False,False,True,False,False,False,False,False,False
1184,3.288373e+09,True,True,True,True,True,True,True,True,4,...,2.0,True,False,False,False,False,False,False,False,False


# Q3

In [8]:
episodes_clean = [key.replace("Star Wars: Episode ", "") for key in EPISODES.keys()]
episodes_clean = [key[key.index(' '):].strip() for key in episodes_clean]

seen_any_df = df_mod[df_mod["seen_any"] == True].copy()
mask = ~seen_any_df[['seen_ep1', 'seen_ep2', 'seen_ep3', 'seen_ep4', 'seen_ep5', 'seen_ep6']].any(axis=1)
seen_any_df = seen_any_df[~mask]

percent_seen = []
for i, name in enumerate(episodes_clean, 1):
    # Use dropna to exclude missing values
    values = seen_any_df[f'seen_ep{i}'].dropna()
    if not values.empty:
        pct = 100 * values.mean()
        percent_seen.append((name, pct))
    else:
        percent_seen.append((name, 0))

plot_df = pd.DataFrame(percent_seen, columns=['Movie', 'Percent Seen'])

plot_df['Movie'] = pd.Categorical(
    plot_df['Movie'],
    categories=episodes_clean[::-1],
    ordered=True
)

In [9]:
plot = ggplot(plot_df, aes(x='Percent Seen', y='Movie')) + \
    geom_bar(stat='identity', fill='#3498db') + \
    ggtitle('Which "Star Wars" Movies Have You Seen?', subtitle="Of 835 respondents who have seen any film") + \
    xlab("Percentage of viewers") + \
    ylab("") + \
    theme_minimal() + \
    ggsize(800, 400)
plot.show()

In [10]:
seen_all_df = df_mod[
    df_mod[['seen_ep1', 'seen_ep2', 'seen_ep3', 'seen_ep4', 'seen_ep5', 'seen_ep6']].all(axis=1)
].copy()

In [ ]:
# seen_all_df.info()

In [11]:
rank_to_ep = {
    f'ep{i}_rank': ep for i, ep in enumerate(episodes_clean, 1)
}

total = len(seen_all_df)
favorite_counts = []

for col, label in rank_to_ep.items():
    count = (seen_all_df[col] == "1").sum()
    pct = 100 * count / total if total > 0 else 0
    favorite_counts.append((label, pct))

plot2_df = pd.DataFrame(favorite_counts, columns=['Movie', 'Percent Ranked Best'])

plot2_df['Movie'] = pd.Categorical(
    plot2_df['Movie'],
    categories=episodes_clean[::-1],
    ordered=True
)

In [12]:
plot2 = ggplot(plot2_df, aes(x='Percent Ranked Best', y='Movie')) + \
    geom_bar(stat='identity', fill='#3498db') + \
    ggtitle('What\'s the Best "Star Wars" Movie?', subtitle="Of 471 respondents who have seen all six films") + \
    xlab("Percent of viewers who ranked it #1") + \
    ylab("") + \
    theme_minimal() + \
    ggsize(800, 400)
plot2.show()

# Q4

In [27]:
model_df = df_mod.copy()
model_df['high_income'] = (model_df['income_hh'] >= 2).astype(int)

# Feature selection
features = [
    'edu', 'is_fan', 'seen_any',
    'seen_ep1', 'seen_ep2', 'seen_ep3',
    'seen_ep4', 'seen_ep5', 'seen_ep6',
    'loc_enc', 'loc_esc', 'loc_mat', 'loc_mtn',
    'loc_neg', 'loc_pcf', 'loc_sat', 'loc_wnc', 'loc_wsc'
]

In [ ]:
X = model_df[features].copy()
y = model_df['high_income']

X = X.fillna(0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

In [36]:
xgbc = XGBClassifier(eval_metric='logloss')
xgbc.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred_xgbc = xgbc.predict(X_test)

print(f"""Accuracy: {accuracy_score(y_test, y_pred_xgbc)}
Classification Report:  
{classification_report(y_test, y_pred_xgbc)}""")

Accuracy: 0.5824915824915825
Classification Report:  
              precision    recall  f1-score   support

           0       0.63      0.60      0.61       163
           1       0.54      0.57      0.55       134

    accuracy                           0.58       297
   macro avg       0.58      0.58      0.58       297
weighted avg       0.58      0.58      0.58       297



In [40]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [41]:
y_pred_lr = lr_model.predict(X_test)

print(f"""Accuracy: {accuracy_score(y_test, y_pred_lr)}
Classification Report:  
{classification_report(y_test, y_pred_lr)}""")

Accuracy: 0.6262626262626263
Classification Report:  
              precision    recall  f1-score   support

           0       0.67      0.64      0.65       163
           1       0.58      0.61      0.60       134

    accuracy                           0.63       297
   macro avg       0.62      0.62      0.62       297
weighted avg       0.63      0.63      0.63       297

